# Clef-flash on Kaggle: Mewndo's second decision backendSpec §37.4 and §37.6 K10. This notebook serves `Cloudflare/clef-flash` on Kaggle's twofree T4s and registers itself with the `mewndo-cloud` gateway, which then sends it the**loose-deadline** calls (`triage`, `receipt`, `showme`) and keeps the free Workers AIneurons for the calls an agent is actually blocked on.Run it with **Save Version -> Save & Run All**, Accelerator **GPU T4 x2**, Internet **on**.A background run lasts up to 12 hours and has no 20-minute idle timeout.**Nothing in this notebook has been run.** The loading path in cell 4, the scoring entrypoint in cell 5 and the per-decision latency are all unverified. See `README.md`.Secrets come from Kaggle Secrets, never from code: `HF_TOKEN`, `GATEWAY_SECRET`, `GATEWAY_URL`.

## 1. The two GPUs this needs

In [ ]:
# T4 has no BF16, so the model is loaded in FP16 spread over both cards (§37.4).# One card is not enough: clef-flash in FP16 is about 18 GB against a T4's 16 GB.import torchassert torch.cuda.is_available(), "turn the GPU accelerator on"n = torch.cuda.device_count()for i in range(n):    p = torch.cuda.get_device_properties(i)    print(f"gpu {i}: {p.name}, {p.total_memory / 1e9:.1f} GB, capability {p.major}.{p.minor}")assert n >= 2, f"need GPU T4 x2, found {n}: set Accelerator to 'GPU T4 x2'"print("torch", torch.__version__)

## 2. Pinned installsEvery version is pinned (§32.5 rule 4) and listed in `docs/versions.md`. `cloudflared` comes from its GitHub release, not from a script piped into a shell.

In [ ]:
# ponytail: pinned, and nothing beyond what cells 4 to 8 actually import.!pip install -q \    transformers==4.57.1 \    accelerate==1.10.1 \    safetensors==0.6.2 \    huggingface_hub==0.36.0 \    fastapi==0.120.1 \    uvicorn==0.38.0CLOUDFLARED_VERSION = "2026.9.1"  # pinned; re-check the release before a demo!wget -q -O /tmp/cloudflared \    https://github.com/cloudflare/cloudflared/releases/download/{CLOUDFLARED_VERSION}/cloudflared-linux-amd64!chmod +x /tmp/cloudflared!/tmp/cloudflared --version

## 3. SecretsAdd these three in **Add-ons -> Secrets** before running. They are never printed and never written to disk.

In [ ]:
from kaggle_secrets import UserSecretsClient_secrets = UserSecretsClient()HF_TOKEN = _secrets.get_secret("HF_TOKEN")GATEWAY_SECRET = _secrets.get_secret("GATEWAY_SECRET")  # shared with the gateway (§37.4)GATEWAY_URL = _secrets.get_secret("GATEWAY_URL").rstrip("/")  # https://<worker>.workers.devassert GATEWAY_URL.startswith("https://"), "the gateway refuses a non-https registration"print("gateway:", GATEWAY_URL)print("secrets loaded:", all(len(s) > 8 for s in (HF_TOKEN, GATEWAY_SECRET)))

## 4. Load clef-flash**Plan A** below: `transformers` with the repo's own code. Print the model card and`joint_schema_model.py` first and read the loading class and the scoring entry point outof them, because neither is documented anywhere this repo can check (§32.5 rule 3).**Plan B**, if plan A fails or a decision takes over 3 s: the community Clef vLLM pluginwith `tensor_parallel_size=2, dtype="float16"`, pinned to the vLLM version its model cardnames, or the W4A16 build on one T4. Record which plan worked in `docs/decisions.md`.

In [ ]:
from huggingface_hub import snapshot_downloadimport pathlibpath = snapshot_download("Cloudflare/clef-flash", token=HF_TOKEN)print("downloaded to", path)# Read before guessing: the scoring entry point is whatever this file exposes.for name in ("README.md", "joint_schema_model.py", "config.json"):    f = pathlib.Path(path) / name    if f.exists():        print(f"\n===== {name} =====")        print(f.read_text()[:4000])    else:        print(f"\n===== {name}: not in the repo =====")

In [ ]:
# Plan A. device_map="auto" spreads the layers over both T4s; float16 because a T4# cannot do bfloat16.from transformers import AutoModel, AutoTokenizerimport torchtokenizer = AutoTokenizer.from_pretrained(path, trust_remote_code=True, token=HF_TOKEN)model = AutoModel.from_pretrained(    path,    trust_remote_code=True,    torch_dtype=torch.float16,    device_map="auto",    token=HF_TOKEN,)model.eval()print(type(model).__name__)print("entry points:", [m for m in dir(model) if "score" in m.lower() or "answer" in m.lower()])print("device map:", getattr(model, "hf_device_map", "n/a"))# Plan B, if the above fails. Uncomment, restart the session, and skip plan A.# !pip install -q vllm==0.11.0 clef-vllm-plugin==0.1.0# from vllm import LLM# model = LLM(model=path, tensor_parallel_size=2, dtype="float16", trust_remote_code=True)

## 5. `systemone(body) -> answers`One call answers every question (§34.3). The shape below is the contract the gatewayalready enforces in `cloud/gateway/src/gateway.js` (`parseClefAnswers`), so get it exactlyright or every answer is thrown away as "unknown shape":- `noul`   -> `{"p_yes": 0..1}`- `score`  -> `{"value": a number inside that question's own `scale`}` — a 0..1 value for a  1..5 scale is **rejected**, deliberately, so a mis-scaled answer can never read as "risk 1"- `choice` -> `{"probabilities": {option: 0..1, ...}}` over exactly that question's `options`;  a bare choice with no distribution is rejected, because §34.4 branches on confidence

In [ ]:
import json, math, redef _prefill(state, question):    """One scoring prompt per question. Replace with the model's own joint-schema call    once cell 4 has shown what it exposes; this is the part most likely to be wrong."""    return json.dumps({"state": state, "question": question}, separators=(",", ":"))def _p_yes(state, question) -> float:    # UNVERIFIED: clef-flash is a System One scorer, so the real path is its own scoring    # method, not generation. Until cell 4 names it, fall back to a single forward pass    # and read a yes/no logit pair.    with torch.inference_mode():        out = model.score(_prefill(state, question)) if hasattr(model, "score") else None    if out is None:        raise RuntimeError("no scoring entry point found; read cell 4's output and fix _p_yes")    return float(out)def systemone(body: dict) -> dict:    state, questions = body["state"], body["questions"]    answers = {}    for q in questions:        if q["type"] == "noul":            answers[q["id"]] = {"p_yes": max(0.0, min(1.0, _p_yes(state, q)))}        elif q["type"] == "score":            scale = q.get("scale") or [1, 2, 3, 4, 5]            low, high = min(scale), max(scale)            # Map one probability onto the question's own scale, never 0..1.            answers[q["id"]] = {"value": low + _p_yes(state, q) * (high - low)}        else:            options = q.get("options") or q.get("choices")            raw = [max(1e-6, _p_yes(state, {**q, "text": f"{q['text']} Answer: {o}"})) for o in options]            total = sum(raw)            answers[q["id"]] = {"probabilities": {o: r / total for o, r in zip(options, raw)}}    return answers

In [ ]:
# Warm up: the first calls pay for CUDA graphs and weight paging, and the gateway's# deadline does not care why a call was slow.WARMUP = {    "state": {"brief": "Fix the failing date tests in api/.", "agent": "claude-code",              "action": {"tool": "Bash", "command": "npm test"}},    "questions": [{"id": "in_scope", "type": "noul", "text": "Is this action part of what the brief asks?"}],}import timefor i in range(3):    t = time.perf_counter()    print(i, systemone(WARMUP), f"{(time.perf_counter() - t) * 1000:.0f} ms")

## 6. Smoke test: 20 Guard cases, 18 must agree`notebooks/evalset/guard20.json`. **If fewer than 18 of 20 agree, the notebook must notregister** (§37.6 K10.6): a backend that disagrees with the rules is worse than no backend,because the gateway would route real decisions to it.

In [ ]:
import json, pathlibEVALSET = pathlib.Path("/kaggle/input/mewndo-evalset/guard20.json")if not EVALSET.exists():    EVALSET = pathlib.Path("notebooks/evalset/guard20.json")  # when the repo is attachedevalset = json.loads(EVALSET.read_text())cases, rule, agreement = evalset["cases"], evalset["pass_rule"], evalset["agreement"]THRESHOLDS = agreement["noul"]["thresholds"]def agrees(expect, got) -> bool:    for qid, want in expect.items():        have = got.get(qid)        if have is None:            return False        if "p_yes" in want:            # Same side of that question's threshold, not the same number.            t = THRESHOLDS.get(qid, 0.5)            if (want["p_yes"] >= t) != (have.get("p_yes", -1) >= t):                return False        elif "value" in want:            if abs(want["value"] - have.get("value", -99)) > 1.0:                return False        elif "choice" in want:            probs = have.get("probabilities") or {}            if not probs or max(probs, key=probs.get) != want["choice"]:                return False    return Trueagreed, disagreed = 0, []for case in cases:    try:        got = systemone(case["request"])    except Exception as e:        disagreed.append((case["id"], f"error: {e}"))        continue    if agrees(case["expect"], got):        agreed += 1    else:        disagreed.append((case["id"], got))print(f"agreed {agreed}/{len(cases)}, need {rule['min_agree']}")for cid, why in disagreed:    print(" disagreed:", cid, str(why)[:200])SMOKE_PASSED = agreed >= rule["min_agree"]assert SMOKE_PASSED, rule["on_fail"]

## 7. The serverFastAPI on `127.0.0.1:8080`: a bearer check against `GATEWAY_SECRET`, an `asyncio.Lock` soGPU calls run one at a time, and a rolling median the gateway uses to decide whether thisbackend is fast enough for a tight deadline (§37.3).

In [ ]:
import asyncio, statistics, threading, timefrom collections import dequefrom fastapi import FastAPI, Header, HTTPException, Requestimport uvicornapp = FastAPI()gpu_lock = asyncio.Lock()latencies = deque(maxlen=50)def p50_ms() -> int:    return int(statistics.median(latencies)) if latencies else 0def check(authorization: str | None):    # The tunnel URL is public, so the bearer token is the only thing in front of the GPU.    if authorization != f"Bearer {GATEWAY_SECRET}":        raise HTTPException(status_code=401, detail="unauthorized")@app.get("/health")async def health():    return {"ok": True, "p50_ms": p50_ms(), "calls": len(latencies)}@app.post("/v1/systemone")async def v1_systemone(request: Request, authorization: str = Header(default=None)):    check(authorization)    body = await request.json()    if not isinstance(body.get("state"), dict) or not body.get("questions"):        raise HTTPException(status_code=400, detail="need state and questions")    started = time.perf_counter()    async with gpu_lock:  # ponytail: one GPU, one call; a queue would only hide the wait        answers = await asyncio.to_thread(systemone, body)    ms = (time.perf_counter() - started) * 1000    latencies.append(ms)    return {"answers": answers, "backend": "kaggle", "ms": int(ms)}def serve():    uvicorn.run(app, host="127.0.0.1", port=8080, log_level="warning")threading.Thread(target=serve, daemon=True).start()time.sleep(3)!curl -s http://127.0.0.1:8080/health

## 8. The tunnelA quick tunnel, so there is no Cloudflare account to configure. Its URL changes on every restart, which is why cell 9 registers rather than hard-codes it. Testers never see this URL.

In [ ]:
import re, subprocess, timedef start_tunnel():    proc = subprocess.Popen(        ["/tmp/cloudflared", "tunnel", "--url", "http://127.0.0.1:8080", "--no-autoupdate"],        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,    )    deadline = time.time() + 60    while time.time() < deadline:        line = proc.stdout.readline()        if not line:            break        found = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)        if found:            return proc, found.group(0)    proc.kill()    raise RuntimeError("cloudflared printed no tunnel URL in 60 s")tunnel, TUNNEL_URL = start_tunnel()print("tunnel up")  # not printed: the URL is the only credential-free way in

## 9. Register, then heartbeat`POST /internal/backend {url, p50_ms}`. The gateway counts this backend dead after three missed 60-second heartbeats (§37.4), so a crashed notebook routes itself out within three minutes.

In [ ]:
import json, urllib.requestdef register(url: str, p50: int) -> bool:    body = json.dumps({"url": url, "p50_ms": p50}).encode()    req = urllib.request.Request(        f"{GATEWAY_URL}/internal/backend",        data=body,        headers={"authorization": f"Bearer {GATEWAY_SECRET}", "content-type": "application/json"},        method="POST",    )    try:        with urllib.request.urlopen(req, timeout=10) as res:            return res.status == 200    except Exception as e:        print("register failed:", e)        return Falseassert SMOKE_PASSED, "refusing to register: the smoke test did not pass"print("registered:", register(TUNNEL_URL, p50_ms()))

## 10. Stay up for 11.5 hours, then stand downThe last cell. It heartbeats every 60 s, restarts `cloudflared` and re-registers if the tunnel dies, and deregisters before Kaggle's 12-hour limit cuts it off mid-decision.

In [ ]:
import timeSTOP_AFTER_S = 11.5 * 3600started = time.time()misses = 0while time.time() - started < STOP_AFTER_S:    time.sleep(60)    if tunnel.poll() is not None:  # the tunnel died: new URL, new registration        print("cloudflared exited; restarting")        try:            tunnel, TUNNEL_URL = start_tunnel()        except Exception as e:            print("could not restart the tunnel:", e)            continue    ok = register(TUNNEL_URL, p50_ms())    misses = 0 if ok else misses + 1    print(f"{int(time.time() - started)}s heartbeat ok={ok} p50={p50_ms()}ms calls={len(latencies)}")    if misses >= 5:        print("the gateway has not accepted 5 heartbeats; stopping")        break# Deregister by pointing the record at a dead URL is NOT possible (the gateway validates# https), so standing down just means stopping: three missed heartbeats and the gateway# marks this backend down by itself (§37.4).tunnel.kill()print("stood down after", int(time.time() - started), "s;", len(latencies), "decisions served")